# Mission 1: Gradient Descent Golf ⛳

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/01_gradient_descent_golf.ipynb)

> **From your syllabus**
> - *Convex Optimization*, Unit I (gradient, Hessian, Taylor) and Unit II (convex functions)
> - *Machine Learning-I*, Unit II (least squares)
> - *Machine Learning-II*, Unit V (SGD)

**Your mission:** a ball sits on a hilly landscape. The hole is at the lowest point. Reach it in as few *strokes* as possible.
Every machine-learning model you will ever meet learns by playing this exact game.

## 0. A 3-minute Python primer
You only need four ideas today. Run each cell with **Shift+Enter**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)           # fixed seed: everyone gets the same numbers

p = np.array([3.0, 2.0])    # a point (x, y) is a NumPy array
print(p, p * 2, p[0])       # arithmetic works on whole arrays; p[0] is the first entry

## 1. The landscape
Our course is the bowl $f(x,y)=x^2+3y^2$. The hole is at $(0,0)$ because $f\ge 0$ and $f(0,0)=0$.

In [ ]:
def f(p):
    x, y = p
    return x**2 + 3 * y**2

print(f(np.array([3.0, 2.0])))   # expect 9 + 12 = 21

## 2. Which way is downhill? (Core 1)
From Convex Optimization Unit I: the gradient points *uphill*, so $-\nabla f$ points downhill.
Work out $\nabla f(x,y)=\left(\frac{\partial f}{\partial x},\frac{\partial f}{\partial y}\right)$ by hand, then fill in the two blanks.

<details><summary>Hint</summary>

$\partial f/\partial x = 2x$ and $\partial f/\partial y = 6y$.
</details>

In [ ]:
def grad_f(p):
    x, y = p
    return np.array([____, ____])   # TODO: replace each ____ with your partial derivative

**Check your maths with the computer.** A finite difference approximates the derivative:
$\frac{f(p+h e_i)-f(p-h e_i)}{2h}\approx \partial_i f(p)$. If your formula is right, the two columns match.

In [ ]:
def numeric_grad(f, p, h=1e-5):
    g = np.zeros_like(p)
    for i in range(len(p)):
        e = np.zeros_like(p); e[i] = h
        g[i] = (f(p + e) - f(p - e)) / (2 * h)
    return g

p = np.array([3.0, 2.0])
print("yours  :", grad_f(p))
print("numeric:", numeric_grad(f, p))
assert np.allclose(grad_f(p), numeric_grad(f, p), atol=1e-4), "Not quite - recheck your partial derivatives!"
print("Gradient correct ✔")

## 3. Take your strokes (Core 2)
One stroke = one step against the gradient with step size (learning rate) $\eta$:

$$p_{t+1}=p_t-\eta\,\nabla f(p_t)$$

Fill in the update line. A stroke counts until you are within 0.01 of the hole (or you give up after 200).

In [ ]:
def play(start, lr, max_strokes=200):
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for stroke in range(max_strokes):
        if np.linalg.norm(p) < 0.01:         # close enough to the hole at (0, 0)
            break
        p = ____                              # TODO: the gradient descent update
        path.append(p.copy())
        if np.linalg.norm(p) > 1e6:           # flew off the course
            break
    return np.array(path)

path = play([3.0, 2.0], lr=0.1)
print("strokes:", len(path) - 1, " final position:", path[-1])

In [ ]:
def draw(paths, labels, title="Gradient Descent Golf"):
    xs, ys = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-3, 3, 200))
    plt.figure(figsize=(6, 4.5))
    plt.contour(xs, ys, xs**2 + 3 * ys**2, levels=15, cmap="Greens")
    for pth, lab in zip(paths, labels):
        plt.plot(pth[:, 0], pth[:, 1], "o-", ms=3, label=lab)
    plt.plot(0, 0, "k*", ms=15, label="hole")
    plt.xlim(-4, 4); plt.ylim(-3, 3); plt.title(title); plt.legend()
    plt.show()

draw([path], [f"lr=0.1 ({len(path)-1} strokes)"])

## 4. Choose your club: the learning rate
Too small is slow, too big overshoots. Try several and compare strokes.

In [ ]:
lrs = [0.02, 0.1, 0.25, 0.32, 0.34]
paths = [play([3.0, 2.0], lr) for lr in lrs]
for lr, pth in zip(lrs, paths):
    status = "diverged!" if np.linalg.norm(pth[-1]) > 1 else f"{len(pth) - 1} strokes"
    print(f"lr = {lr:<5}  {status}")
draw(paths[:4], [f"lr={lr}" for lr in lrs[:4]], "Four clubs")

**Challenge:** which learning rate gives the fewest strokes? Guess before reading the maths box below. 🏆
Post your best score in the group chat.

In [ ]:
# Interactive club picker (works in Jupyter and Colab; if sliders do not show, just edit lr and re-run)
try:
    from ipywidgets import interact, FloatSlider
    def try_lr(lr=0.1):
        pth = play([3.0, 2.0], lr)
        draw([pth], [f"lr={lr} ({len(pth)-1} strokes)"])
    interact(try_lr, lr=FloatSlider(min=0.01, max=0.4, step=0.01, value=0.1))
except Exception as e:
    print("Widgets unavailable:", e)

## Math box: why does it work?
For $f(p)=\tfrac12 p^\top H p$ with $H=\mathrm{diag}(2,6)$, one stroke gives
$$p_{t+1}=(I-\eta H)\,p_t .$$
Each coordinate shrinks by the factor $|1-\eta\lambda_i|$, where $\lambda_i\in\{2,6\}$ are the **Hessian eigenvalues**
(Convex Optimization Unit I). The ball converges iff $|1-\eta\lambda_i|<1$ for all $i$, i.e.
$$0<\eta<\frac{2}{\lambda_{\max}}=\frac{2}{6}\approx 0.333,$$
which is why $0.34$ diverged. The *best* $\eta$ balances the worst two factors:
$\eta^\star=\dfrac{2}{\lambda_{\min}+\lambda_{\max}}=0.25$. The code below verifies this numerically.

In [ ]:
H = np.diag([2.0, 6.0])
eigs = np.linalg.eigvalsh(H)
print("Hessian eigenvalues:", eigs)
print("stability limit 2/lambda_max =", 2 / eigs.max())
print("best lr 2/(lmin+lmax)        =", 2 / eigs.sum())

lr_grid = np.linspace(0.01, 0.33, 33)
strokes = [len(play([3.0, 2.0], lr)) - 1 for lr in lr_grid]
print("empirical best lr:", lr_grid[int(np.argmin(strokes))], "with", min(strokes), "strokes")
plt.plot(lr_grid, strokes); plt.xlabel("learning rate"); plt.ylabel("strokes to hole"); plt.show()

## Stretch: a narrow valley
Real loss functions are often like $g(x,y)=x^2+30\,y^2$: eigenvalues far apart (a large **condition number**).
Plain gradient descent zig-zags. **Momentum** keeps a running velocity:
$$v_{t+1}=\beta v_t-\eta\nabla g(p_t),\qquad p_{t+1}=p_t+v_{t+1}$$
Fill in the update and compare strokes for different $\beta$. Surprise: bigger is not always better. Why might $\beta=0.9$ overshoot?

In [ ]:
def grad_g(p):
    return np.array([2 * p[0], 60 * p[1]])

def play_momentum(start, lr, beta, max_strokes=500):
    p = np.array(start, dtype=float); v = np.zeros(2); n = 0
    for n in range(max_strokes):
        if np.linalg.norm(p) < 0.01:
            break
        v = ____                 # TODO: velocity update
        p = ____                 # TODO: position update
    return n, p

for beta in [0.0, 0.5, 0.9]:
    print("beta =", beta, "->", play_momentum([3.0, 1.0], lr=0.03, beta=beta))

## Quest (take-home)
Gradient descent is how a model *fits data*. Generate points $y\approx 2x+1+\text{noise}$, define the loss
$L(w,b)=\frac1n\sum(wx_i+b-y_i)^2$, derive $\nabla L$, run your `play` loop on $(w,b)$, and compare with the closed-form
normal equation $\hat\theta=(X^\top X)^{-1}X^\top y$ from *ML-I Unit II*. Post your plot on GitHub for the Hall of Fame!

## You just used...
- the **gradient** and the **Hessian** (Convex Optimization, Unit I)
- the **eigenvalue condition** $|1-\eta\lambda|<1$ for convergence of a linear iteration
- **gradient descent**, the engine inside logistic regression, neural nets and SGD (ML-I, ML-II)

## Your CV line
> *Implemented gradient descent from scratch in Python (NumPy), verified gradients by finite differences and analysed
> convergence via Hessian eigenvalues; demonstrated momentum on ill-conditioned problems.*